# VS Code <-> Google Colab Baglantisi

Bu notebook, Google Colab uzerinde bir SSH sunucusu ve **cloudflared** tuneli baslatir.  
Ardindan VS Code'daki **Remote - SSH** eklentisi ile Colab'a baglanabilirsiniz.

> **Gereksinimler (yerel bilgisayarinizda):**
> - [VS Code](https://code.visualstudio.com/) kurulu olmali
> - [Remote - SSH](https://marketplace.visualstudio.com/items?itemName=ms-vscode-remote.remote-ssh) eklentisi kurulu olmali
> - [cloudflared](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/downloads/) kurulu olmali

## Adim 1 - Bagimliliklar kur

In [ ]:
import subprocess, os

# OpenSSH sunucusunu kur
subprocess.run(['apt-get', 'install', '-qq', '-y', 'openssh-server'], check=True)

# cloudflared'i indir ve kur
subprocess.run([
    'wget', '-q',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb',
    '-O', '/tmp/cloudflared.deb'
], check=True)
subprocess.run(['dpkg', '-i', '/tmp/cloudflared.deb'], check=True)

print('Kurulum tamamlandi.')

## Adim 2 - SSH sifresi ve anahtarini ayarla

In [ ]:
import getpass, re as _re

# root kullanicisi icin sifre belirle
password = getpass.getpass('Colab SSH sifresi girin: ')
os.system(f"echo 'root:{password}' | chpasswd")

# SSH sunucusunu yapilandir:
# Mevcut carpisan direktifleri yorum satirina al, ardindan yeni degerleri ekle
overrides = {
    'PermitRootLogin': 'yes',
    'PasswordAuthentication': 'yes',
    'X11Forwarding': 'yes',
}
with open('/etc/ssh/sshd_config', 'r') as _f:
    _cfg = _f.read()
for _key, _val in overrides.items():
    _cfg = _re.sub(rf'^({_key}\\s+.*)', r'#\\1', _cfg, flags=_re.MULTILINE)
    _cfg += f'\n{_key} {_val}'
with open('/etc/ssh/sshd_config', 'w') as _f:
    _f.write(_cfg)

# SSH host anahtarlarini olustur ve sunucuyu baslatma
subprocess.run(['ssh-keygen', '-A'], check=True)
subprocess.Popen(['/usr/sbin/sshd', '-D'])

print('SSH sunucusu baslatildi.')

## Adim 3 - Cloudflared tunelini baslatma

In [ ]:
import threading, time, re

# Thread-safe URL tespiti icin Event kullan
_tunnel_ready = threading.Event()
_tunnel_url_holder = [None]
_tunnel_log = []

def _run_tunnel():
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'tcp://localhost:22'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        _tunnel_log.append(line.strip())
        match = re.search(r'(https://[\w.-]+\.trycloudflare\.com)', line)
        if match and not _tunnel_ready.is_set():
            _tunnel_url_holder[0] = match.group(1)
            _tunnel_ready.set()

t = threading.Thread(target=_run_tunnel, daemon=True)
t.start()

# Tunel URL'sinin hazir olmasini bekle (en fazla 30 saniye)
_tunnel_ready.wait(timeout=30)
tunnel_url = _tunnel_url_holder[0]

if tunnel_url:
    host = tunnel_url.replace('https://', '')
    print('Tunel aktif!')
    print(f'\nTunel adresi : {tunnel_url}')
    print('\nVS Code baglanti komutu (~/.ssh/config):')
    print(f'''
Host colab
    HostName {host}
    User root
    Port 22
    ProxyCommand cloudflared access ssh --hostname %h
''')
    print('Yukaridaki bloguu ~/.ssh/config dosyaniza ekleyin,')
    print('ardindan VS Code Remote-SSH ile "colab" hostuna baglanin.')
else:
    print('Tunel baslatilmadi. Gunlukler:')
    print('\n'.join(_tunnel_log[-20:]))

## Adim 4 - Oturumu acik tut

Asagidaki hucre, Colab oturumunun zaman asimina ugramamasi icin calisir durumda tutulur.  
VS Code baglantisinii kapattiktan sonra bu hucreyi de durdurabilirsiniz.

In [ ]:
import time
print('Oturum acik tutuluyor... (durdurmak icin hucreyi kesin)')
while True:
    time.sleep(60)